# If you just updated `spswarehouse` you'll need to update `scopes` in `credentials.py` (Line 23) from `credentials.py.template` before using GoogleDrive.

### GoogleDrive is a wrapper on pydrive.drive (https://pythonhosted.org/PyDrive/)

In [ ]:
import spswarehouse
from spswarehouse.table_utils import *
from spswarehouse.warehouse import Warehouse
from spswarehouse.googledrive import GoogleDrive

import pandas as pd
import os

# Getting a list of available files

In [ ]:
# This cell is commented out because it takes a VERY long time to run if your service account has been in use for a while.


# Retrieve all non-trashed files
# fileList = GoogleDrive.ListFile({'q': "trashed = False"}).GetList()

# 'q' is a query parameter; see next cell for what fields can be queried
# Other passable parameters: https://developers.google.com/drive/api/v3/reference/files/list
# for file in fileList:
#     print('Title: %s, ID: %s' % (file['title'], file['id']))

In [ ]:
# Search for file by title
fileList2 = GoogleDrive.ListFile({'q': "title contains 'SY19 - Final'"}).GetList()
for file in fileList2:
    print('Title: %s, ID: %s' % (file['title'], file['id']))

In [ ]:
# Search for files in a folder
folderID = '1NtuZczi7zPQqXAKcd60yQtzPDa6q3jA-'
# You can get the folder id either by listing all files as above (folders show up in the list)
# or by opening the folder in a browser and copying the string after `folder/`

folderFiles = GoogleDrive.ListFile({'q': f"'{folderID}' in parents"}).GetList()
for file in folderFiles:
    print('Title: %s, ID: %s' % (file['title'], file['id']))

In [ ]:
# Sample file parameters that can be queried
# Some of these parameters are actually dictionaries, and contain additional parameters that can be queried
# e.g., `trashed` is a parameter in `labels`

# I highly recommend that you stick to querying title or all files

file = folderFiles[-1]
dict(file)

# Retrieving a file

In [ ]:
# Option 1: Use the id
# This is preferred, because it's guaranteed to return a single file

# You can get the id either by using the list above 
# OR by clicking on the file in drive, `Get shareable link`, then copying everything after `id=`
# e.g., https://drive.google.com/open?id=1Mug7698RhjfrVrIgHs6Yc3AM0mhGNA7q


fileID = '1Mug7698RhjfrVrIgHs6Yc3AM0mhGNA7q' #This is the an old csv version of sites_historical
# The id of the file can also be retrieved via `file['id']` (See the print statement in the file search examples)

sampleFile = GoogleDrive.CreateFile({'id': fileID})
print(sampleFile['title'])

In [ ]:
# Option 2: Do a title search
# This can get messy if there are multiple files with the same name

titleSampleFile = GoogleDrive.ListFile({'q': "title = 'spswarehouse_sample_file.csv'"}).GetList()[0]
print(titleSampleFile['title'])

In [ ]:
# Downloading the file to your local machine as a file called <filename>
filename = 'data.csv'
sampleFile.GetContentFile(filename)

In [ ]:
# Retrieve file contents as a string, which you can then manipulate
dataString = sampleFile.GetContentString()
dataString[:500]

# Uploading to Warehouse from Drive

In [ ]:
schema = 'wild_west'
table_name = 'google_drive_test'

create_sql = create_table_stmt(table_name, schema=schema, google_drive_id = fileID)
print(create_sql)

In [ ]:
Warehouse.execute(create_sql)

In [ ]:
Warehouse.upload_google_drive_csv(table_name, schema, google_drive_id = fileID)

# Cleanup of this notebook

#### Run this when you're done to clean up loose ends.

In [ ]:
Warehouse.execute(f"DROP TABLE IF EXISTS {schema}.{table_name}")
if os.path.exists(filename):
    os.remove(filename)